<a href="https://colab.research.google.com/github/sparshap95-tech/NLP/blob/main/Simple_RNN_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
corpus = [
    "I love machine learning",
    "wordvec is great algorithm",
    "Implementing word2vec is fun"
]

In [3]:
sentences = [s.lower().split() for s in corpus]


In [4]:

vocab = sorted(set(word for sent in sentences for word in sent))
word2idx = {w:i for i,w in enumerate(vocab)}
idx2word = {i:w for w,i in word2idx.items()}

In [5]:

vocab_size = len(vocab)

In [6]:

word2idx

{'algorithm': 0,
 'fun': 1,
 'great': 2,
 'i': 3,
 'implementing': 4,
 'is': 5,
 'learning': 6,
 'love': 7,
 'machine': 8,
 'word2vec': 9,
 'wordvec': 10}

In [7]:

X = []
Y = []
for sent in sentences:
    for i in range(len(sent)-1):
        X.append(word2idx[sent[i]])
        Y.append(word2idx[sent[i+1]])

In [8]:

X = torch.tensor(X)
Y = torch.tensor(Y)

In [9]:
class NextWordRNN(nn.Module):

    def __init__(self, vocab_size, embed_dim=10, hidden_dim=16):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):

        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])

        return out

In [10]:
model = NextWordRNN(vocab_size)


In [11]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [12]:
for epoch in range(300):

    optimizer.zero_grad()

    inputs = X.unsqueeze(1)
    outputs = model(inputs)

    loss = loss_fn(outputs, Y)

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 2.5272433757781982
Epoch: 50 Loss: 0.19651463627815247
Epoch: 100 Loss: 0.16538646817207336
Epoch: 150 Loss: 0.16053326427936554
Epoch: 200 Loss: 0.1583518534898758
Epoch: 250 Loss: 0.15713147819042206


In [13]:
def predict_next(word):
    model.eval()
    idx = torch.tensor([[word2idx[word.lower()]]])

    with torch.no_grad():
        out = model(idx)
        pred = torch.argmax(out).item()

    return idx2word[pred]

In [14]:


print(predict_next("machine"))
print(predict_next("is"))
print(predict_next("wordvec"))

learning
great
is
